In [1]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.appName('income.csv').getOrCreate()
df = spark.read.csv(r'C:\Users\user\Documents\Data Science in practice\income.csv', header = True, inferSchema= True)


#### Preview

In [2]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- weight: double (nullable = true)
 |-- education: string (nullable = true)
 |-- education_years: double (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital_gain: double (nullable = true)
 |-- capital_loss: double (nullable = true)
 |-- hours_per_week: double (nullable = true)
 |-- citizenship: string (nullable = true)
 |-- income_class: string (nullable = true)



In [3]:
df.show()

+---+-----------------+--------+-------------+---------------+--------------------+------------------+--------------+-------------------+-------+------------+------------+--------------+--------------+------------+
|age|        workclass|  weight|    education|education_years|      marital_status|        occupation|  relationship|               race|    sex|capital_gain|capital_loss|hours_per_week|   citizenship|income_class|
+---+-----------------+--------+-------------+---------------+--------------------+------------------+--------------+-------------------+-------+------------+------------+--------------+--------------+------------+
| 39|        State-gov| 77516.0|    Bachelors|           13.0|       Never-married|      Adm-clerical| Not-in-family|              White|   Male|      2174.0|         0.0|          40.0| United-States|       <=50K|
| 50| Self-emp-not-inc| 83311.0|    Bachelors|           13.0|  Married-civ-spouse|   Exec-managerial|       Husband|              White|   

#### OneHotEncoder and VectorAssemble

In [4]:
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

df = df.withColumn("age", col("age").cast("int"))
df = df.withColumn("weight", col("weight").cast("int"))
df = df.withColumn("education_years", col("education_years").cast("int"))
df = df.withColumn("capital_gain", col("capital_gain").cast("int"))
df = df.withColumn("capital_loss", col("capital_loss").cast("int"))
df = df.withColumn("hours_per_week", col("hours_per_week").cast("int"))

categorical_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "citizenship"
]

numeric_cols = [
    "age",
    "weight",
    "education_years",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]


workclass_indexer = StringIndexer(inputCol="workclass", outputCol="workclass_index")
education_indexer = StringIndexer(inputCol="education", outputCol="education_index")
marital_indexer = StringIndexer(inputCol="marital_status", outputCol="marital_status_index")
occupation_indexer = StringIndexer(inputCol="occupation", outputCol="occupation_index")
relationship_indexer = StringIndexer(inputCol="relationship", outputCol="relationship_index")
race_indexer = StringIndexer(inputCol="race", outputCol="race_index")
sex_indexer = StringIndexer(inputCol="sex", outputCol="sex_index")
citizenship_indexer = StringIndexer(inputCol="citizenship", outputCol="citizenship_index")



encoder = OneHotEncoder(
    inputCols=[
        "workclass_index",
        "education_index",
        "marital_status_index",
        "occupation_index",
        "relationship_index",
        "race_index",
        "sex_index",
        "citizenship_index"
    ],
    outputCols=[
        "workclass_vec",
        "education_vec",
        "marital_status_vec",
        "occupation_vec",
        "relationship_vec",
        "race_vec",
        "sex_vec",
        "citizenship_vec"
    ]
)

In [5]:
label_indexer = StringIndexer(inputCol="income_class", outputCol="label")


assembler = VectorAssembler(
    inputCols=[
        "age",
        "weight",
        "education_years",
        "capital_gain",
        "capital_loss",
        "hours_per_week",
        "workclass_vec",
        "education_vec",
        "marital_status_vec",
        "occupation_vec",
        "relationship_vec",
        "race_vec",
        "sex_vec",
        "citizenship_vec"
    ],
    outputCol="features"
)

pipeline = Pipeline(stages=[
    workclass_indexer,
    education_indexer,
    marital_indexer,
    occupation_indexer,
    relationship_indexer,
    race_indexer,
    sex_indexer,
    citizenship_indexer,
    encoder,
    label_indexer,
    assembler
])


indexed_df = pipeline.fit(df).transform(df)

indexed_df.select("features", "label").show(truncate=False) 


+---------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                 |label|
+---------------------------------------------------------------------------------------------------------+-----+
|(100,[0,1,2,3,5,10,16,30,38,50,54,58,59],[39.0,77516.0,13.0,2174.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0.0  |
|(100,[0,1,2,5,7,16,29,37,49,54,58,59],[50.0,83311.0,13.0,13.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])          |0.0  |
|(100,[0,1,2,5,6,14,31,44,50,54,58,59],[38.0,215646.0,9.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])          |0.0  |
|(100,[0,1,2,5,6,19,29,44,49,55,58,59],[53.0,234721.0,7.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])          |0.0  |
|(100,[0,1,2,5,6,16,29,35,53,55,68],[28.0,338409.0,13.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                |0.0  |
|(100,[0,1,2,5,6,17,29,37,53,54,59],[37.0,284582.0,14.0,40.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0

#### Data Split

In [6]:
train, test = indexed_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count()}")
print(f"Test rows: {test.count()}")

Train rows: 26076
Test rows: 6485


#### DecisionTreeClassifier and RandomForestClassifier

In [7]:
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier


decisiontree = DecisionTreeClassifier(featuresCol="features", labelCol="label")


model = decisiontree.fit(train)


predictions = model.transform(test)

predictions.select("label", "prediction").show()

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
+-----+----------+
only showing top 20 rows


In [8]:


rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100)

rf_model = rf.fit(train)

rf_predictions = rf_model.transform(test)

rf_predictions.select("label", "prediction").show(10)

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
+-----+----------+
only showing top 10 rows


#### Evaluation

In [9]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

In [10]:
dt_accuracy = evaluator.evaluate(predictions)
dt_f1 = f1_evaluator.evaluate(predictions)
dt_precision = precision_evaluator.evaluate(predictions)
dt_recall = recall_evaluator.evaluate(predictions)

In [11]:
rf_accuracy = evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)
rf_precision = precision_evaluator.evaluate(rf_predictions)
rf_recall = recall_evaluator.evaluate(rf_predictions)

In [12]:
results = [
    ("Decision Tree", dt_accuracy, dt_precision, dt_recall, dt_f1),
    ("Random Forest", rf_accuracy, rf_precision, rf_recall, rf_f1)
]

results_df = spark.createDataFrame(
    results,
    ["Model", "Accuracy", "Precision", "Recall", "F1 Score"]
)

results_df.show(truncate=False)

+-------------+------------------+------------------+------------------+------------------+
|Model        |Accuracy          |Precision         |Recall            |F1 Score          |
+-------------+------------------+------------------+------------------+------------------+
|Decision Tree|0.8431765612952968|0.8357342910254923|0.8431765612952968|0.8314989456490177|
|Random Forest|0.8251349267540478|0.8219977909973499|0.8251349267540479|0.7999354305586464|
+-------------+------------------+------------------+------------------+------------------+

